# Deep search (shift-and-stack)

**Goal:** find asteroids too faint to see in any single frame, by adding frames along trial motions, and prove which
detections are real. Read `docs/deep_search.md` alongside this notebook.

The three heavy runs (real search, null test, injected fakes) take ~15 minutes each on an M3 Pro, so they are cached in
`data/processed/deep_search__<unit tag>__<run>.pkl`. Delete a file to recompute it.

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from asthunt.config.load_config import load_config
from asthunt.acquire.load_unit_metadata import load_unit_metadata
from asthunt.acquire.select_night_exposures import select_night_exposures
from asthunt.dataset.make_unit_tag import make_unit_tag
from asthunt.pipeline.run_deep_search import run_deep_search
from asthunt.deep.measure_velocity_consistency import measure_velocity_consistency
from asthunt.crossmatch.crossmatch_deep_tracklets import crossmatch_deep_tracklets
from asthunt.simulate.plan_injections import plan_injections
from asthunt.evaluate.measure_completeness import measure_completeness
from asthunt.evaluate.find_magnitude_at_completeness import find_magnitude_at_completeness

project = load_config("project"); paths = project["paths"]
acquire, deep = load_config("acquire"), load_config("deep")
unit = {"field": 618, "ccd": 3, "quadrant": 4, "night": "2019-02-08"}
metadata = load_unit_metadata(unit, paths["raw"])
frames = select_night_exposures(metadata, unit["night"], acquire["filter"], 10_000)
print(len(frames), "good frames over", round((frames["obsjd"].max() - frames["obsjd"].min()) * 24, 1), "hours")


def cached_run(name, **options):
    path = paths["processed"] / f"deep_search__{make_unit_tag(unit)}__{name}.pkl"
    if path.exists():
        result = pickle.load(open(path, "rb"))
        if len(result["tracklets"]) and "consistent" not in result["tracklets"]:
            # cached before the velocity check existed: it only needs data that is already in the cache
            result["tracklets"] = measure_velocity_consistency(result["tracklets"], result["members"], result["stack_detections"])
            result["tracklets"]["consistent"] = result["tracklets"]["velocity_mismatch"] <= deep["max_velocity_mismatch_pixels_per_hour"]
        return result
    result = run_deep_search(unit, metadata, acquire, deep, paths, device="mps", progress=print, **options)
    result.pop("reference_wcs")
    if "fakes" in options:
        result["fakes"] = options["fakes"]
    pickle.dump(result, open(path, "wb"))
    return result

## 1. The real search
Steps: SNR maps → standardise every pixel over the night → hide what single frames already see → stack 8-frame groups along
4,888 trial motions → merge peaks → link across stacks → velocity-consistency check.

In [ ]:
real = cached_run("real")
print(real["stacks"][["stack", "frames", "peaks"]].to_string(index=False))
matched = crossmatch_deep_tracklets(real, unit, paths)
consistent = matched[matched["consistent"]]
print(f"\n{len(matched)} tracklets; consistent {len(consistent)}: known {int((consistent['known_name'] != '').sum())}, "
      f"unknown {int((consistent['known_name'] == '').sum())}")

## 2. The null test: how many false tracklets should we expect?
Same search, frame times shuffled. Real objects no longer line up; noise and artifacts are unchanged.

In [ ]:
null = cached_run("null", shuffle_times_seed=1)
null_tracklets = null["tracklets"]
print(f"null test: {len(null_tracklets)} tracklets, consistent: {int(null_tracklets['consistent'].sum())}")

plt.hist(matched.loc[matched["known_name"] != "", "velocity_mismatch"], bins=30, range=(0, 120), alpha=0.6, label="known asteroids")
plt.hist(null_tracklets["velocity_mismatch"], bins=30, range=(0, 120), alpha=0.6, label="null test (all false)")
plt.axvline(deep["max_velocity_mismatch_pixels_per_hour"], c="k", ls="--", label="cut")
plt.xlabel("velocity mismatch (px/h)"); plt.legend(); plt.show()

## 3. Depth: injection-recovery with 300 fakes (mag 20-23)

In [ ]:
rng = np.random.default_rng(2024)
fakes = plan_injections(300, (3080, 3072), frames["obsjd"].tolist(), (20.0, 23.0), (5.0, 100.0), 1.012, 150, rng)
injected = cached_run("inject", fakes=fakes, rng=rng)
fakes = injected["fakes"]
times = frames["obsjd"].to_numpy()

lines = {}
for fake_id, track in fakes.groupby("injection_id"):
    hours = (times[track["frame_index"].to_numpy()] - times[0]) * 24
    lines[fake_id] = (np.polyfit(hours, track["x"], 1), np.polyfit(hours, track["y"], 1))

found, mean_snr = set(), {}
tracklets, members, detections = injected["tracklets"], injected["members"], injected["stack_detections"]
for _, tracklet in tracklets[tracklets["consistent"]].iterrows():
    rows = detections.loc[members.loc[members["tracklet_id"] == tracklet["tracklet_id"], "detection_id"]]
    hours = (rows["obsjd"].to_numpy() - times[0]) * 24
    for fake_id, ((ax, bx), (ay, by)) in lines.items():
        distance = np.hypot(rows["x_common"].to_numpy() - (ax * hours + bx), rows["y_common"].to_numpy() - (ay * hours + by))
        if (distance < 3).sum() >= 3:
            found.add(fake_id); mean_snr[fake_id] = rows["snr"].mean(); break

per_fake = fakes.groupby("injection_id").agg(mag=("mag", "first")).reset_index()
on_chip = fakes[fakes["x"].between(30, 3040) & fakes["y"].between(30, 3050)].groupby("injection_id").size()
per_fake = per_fake[per_fake["injection_id"].isin(on_chip[on_chip >= 20].index)]
per_fake["found"] = per_fake["injection_id"].isin(found)
curve = measure_completeness(per_fake, "found", "mag", np.arange(20, 23.25, 0.25))
plt.errorbar(curve["bin_centre"], curve["completeness"], curve["uncertainty"], fmt="o-", label="deep search")
plt.axvline(21.41, ls="--", c="gray", label="v4 single-frame pipeline, 50% (same night)")
plt.xlabel("magnitude"); plt.ylabel("completeness"); plt.legend(); plt.show()
print(f"deep search 50% completeness: mag {find_magnitude_at_completeness(curve, 0.5):.2f}")

Zero completeness for bright fakes is by design: anything already above 5σ in a single frame is hidden from the stack (to
prevent echoes) and left to the regular pipeline. The two searches together cover everything down to ~22.1.

## 4. Candidates: unknown, consistent, and calibrated

In [ ]:
calibration = per_fake[per_fake["found"]].assign(snr=lambda t: t["injection_id"].map(mean_snr))
slope, intercept = np.polyfit(np.log10(calibration["snr"]), calibration["mag"], 1)
stack_snr = real["members"].merge(real["stack_detections"][["snr"]], left_on="detection_id", right_index=True).groupby("tracklet_id")["snr"].mean()
candidates = consistent[consistent["known_name"] == ""].copy()
candidates["mean_stack_snr"] = candidates["tracklet_id"].map(stack_snr)
candidates["estimated_mag"] = intercept + slope * np.log10(candidates["mean_stack_snr"])
print(f"calibration from {len(calibration)} found fakes: mag = {intercept:.2f} + {slope:.2f} log10(SNR)")
candidates[["tracklet_id", "n_detections", "rate_arcsec_per_hour", "max_residual", "velocity_mismatch", "mean_stack_snr", "estimated_mag"]].round(2)

## Exercises
1. Turn off `mask_bright_sigma`. How many unknown tracklets appear, and how close are they to known asteroids?
2. Run the null test with another seed. How much does the false-tracklet count vary?
3. Why is a single night not enough to report these candidates as discoveries?